In [1]:
import pandas as pd
from io import StringIO

csv_data = """order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10,Paid
RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0,paid
RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,,Pending
RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
RT-1005,2026-01-09,Fresher,,Mentor Session,1,999,0,Failed
RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10,Paid
RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105,Paid
RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0,Pending
RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0,Paid
RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15,Refunded
RT-1011,,Student,Kochi,Mentor Session,1,999,0,Paid"""

df = pd.read_csv(StringIO(csv_data))

df.head()

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [2]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Number of rows: 12
Number of columns: 9

Column names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Data types:
order_id             object
order_date           object
customer_segment     object
city                 object
category             object
quantity             object
unit_price            int64
discount_pct        float64
payment_status       object
dtype: object


In [3]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64


In [4]:
duplicate_rows = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 1


In [5]:
quantity_numeric = pd.to_numeric(df["quantity"], errors="coerce")

invalid_quantity = df[
    quantity_numeric.isna() | (quantity_numeric <= 0)
]

print("Invalid quantity records:")
print(invalid_quantity[["order_id", "quantity"]])

Invalid quantity records:
  order_id quantity
6  RT-1006       -1
8  RT-1008      two


In [6]:
discount_numeric = pd.to_numeric(df["discount_pct"], errors="coerce")

invalid_discount = df[
    discount_numeric.notna() &
    ((discount_numeric < 0) | (discount_numeric > 100))
]

print("Invalid discount records:")
print(invalid_discount[["order_id", "discount_pct"]])

Invalid discount records:
  order_id  discount_pct
7  RT-1007         105.0


In [7]:
date_check = pd.to_datetime(df["order_date"], errors="coerce")

invalid_dates = df[
    date_check.isna()
]

print("Invalid or missing date records:")
print(invalid_dates[["order_id", "order_date"]])

Invalid or missing date records:
   order_id  order_date
1   RT-1002  03/01/2026
6   RT-1006  2026-13-10
11  RT-1011         NaN


In [8]:
allowed_status = ["Paid", "Pending", "Failed", "Refunded"]

invalid_status = df[
    ~df["payment_status"].str.strip().str.title().isin(allowed_status)
]

print("Invalid payment status records:")
print(invalid_status[["order_id", "payment_status"]])

Invalid payment status records:
Empty DataFrame
Columns: [order_id, payment_status]
Index: []


In [9]:
allowed_categories = [
    "Learning Kit",
    "Course Access",
    "Mentor Session"
]

invalid_category = df[
    ~df["category"].str.strip().isin(allowed_categories)
]

print("Invalid category records:")
print(invalid_category[["order_id", "category"]])

Invalid category records:
Empty DataFrame
Columns: [order_id, category]
Index: []


In [10]:
allowed_segments = [
    "Student",
    "Fresher",
    "Professional"
]

invalid_segment = df[
    ~df["customer_segment"].str.strip().str.title().isin(allowed_segments)
]

print("Invalid customer segment records:")
print(invalid_segment[["order_id", "customer_segment"]])

Invalid customer segment records:
Empty DataFrame
Columns: [order_id, customer_segment]
Index: []


In [11]:
invalid_city = df[
    df["city"].isna() | (df["city"].str.strip() == "")
]

print("Missing or empty city records:")
print(invalid_city[["order_id", "city"]])

Missing or empty city records:
  order_id city
5  RT-1005  NaN


In [12]:
date_check = pd.to_datetime(df["order_date"], errors="coerce")

print("Latest valid order date:", date_check.max())
print("Today:", pd.Timestamp.today().normalize())

days_old = (pd.Timestamp.today().normalize() - date_check.max()).days
print("Days since latest order:", days_old)


Latest valid order date: 2026-01-18 00:00:00
Today: 2026-09-11 00:00:00
Days since latest order: 236


In [13]:
print("===== DATA QUALITY SUMMARY =====")

print("\n1. Completeness")
print("Missing values:")
print(df.isnull().sum())

print("\n2. Uniqueness")
print("Duplicate rows:", df.duplicated().sum())

print("\n3. Quantity Validity")
print("Invalid quantity records:",
      (pd.to_numeric(df["quantity"], errors="coerce").isna() |
       (pd.to_numeric(df["quantity"], errors="coerce") <= 0)).sum())

print("\n4. Discount Validity")
discount = pd.to_numeric(df["discount_pct"], errors="coerce")
print("Invalid discount records:",
      ((discount < 0) | (discount > 100)).sum())

print("\n5. Date Validity")
dates = pd.to_datetime(df["order_date"], errors="coerce")
print("Invalid/missing dates:", dates.isna().sum())

print("\n6. Payment Status")
print(df["payment_status"].value_counts(dropna=False))

print("\n7. Customer Segment")
print(df["customer_segment"].value_counts(dropna=False))

print("\n8. Category")
print(df["category"].value_counts(dropna=False))

print("\n9. Latest Valid Order Date")
print(dates.max())

===== DATA QUALITY SUMMARY =====

1. Completeness
Missing values:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64

2. Uniqueness
Duplicate rows: 1

3. Quantity Validity
Invalid quantity records: 2

4. Discount Validity
Invalid discount records: 1

5. Date Validity
Invalid/missing dates: 3

6. Payment Status
payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1
Name: count, dtype: int64

7. Customer Segment
customer_segment
Student         4
Professional    4
Fresher         3
student         1
Name: count, dtype: int64

8. Category
category
Learning Kit      5
Course Access     4
Mentor Session    3
Name: count, dtype: int64

9. Latest Valid Order Date
2026-01-18 00:00:00


# Retail Orders – Data Quality Profile

## Project Objective
This notebook profiles the retail orders dataset and identifies data quality issues related to completeness, uniqueness, validity, consistency, and freshness.

## 1. Dataset Overview

The dataset contains retail order information including order date, customer segment, city, product category, quantity, unit price, discount percentage, and payment status.

In [14]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows: 12
Columns: 9

Column Names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Data Types:
order_id             object
order_date           object
customer_segment     object
city                 object
category             object
quantity             object
unit_price            int64
discount_pct        float64
payment_status       object
dtype: object


## 2. Completeness Check

This check identifies missing values in each field of the dataset.

In [15]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64


## 3. Uniqueness Check

This check identifies duplicate records and verifies the uniqueness of order IDs.

In [16]:
duplicate_rows = df.duplicated().sum()
duplicate_order_ids = df["order_id"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate order IDs:", duplicate_order_ids)

Duplicate rows: 1
Duplicate order IDs: 1


## 4. Validity Checks

This section checks whether important fields follow the defined data quality rules.

In [17]:
# Quantity validity
quantity_numeric = pd.to_numeric(df["quantity"], errors="coerce")
invalid_quantity = df[
    quantity_numeric.isna() | (quantity_numeric <= 0)
]

# Discount validity
discount_numeric = pd.to_numeric(df["discount_pct"], errors="coerce")
invalid_discount = df[
    discount_numeric.notna() &
    ((discount_numeric < 0) | (discount_numeric > 100))
]

# Date validity
date_check = pd.to_datetime(df["order_date"], errors="coerce")
invalid_dates = df[
    date_check.isna()
]

print("=== Quantity Validity ===")
print("Invalid quantity records:", len(invalid_quantity))

print("\n=== Discount Validity ===")
print("Invalid discount records:", len(invalid_discount))

print("\n=== Date Validity ===")
print("Invalid or missing date records:", len(invalid_dates))

=== Quantity Validity ===
Invalid quantity records: 2

=== Discount Validity ===
Invalid discount records: 1

=== Date Validity ===
Invalid or missing date records: 3


## 5. Consistency Checks

This section checks whether categorical values use consistent formatting and standard values.

In [18]:
# Payment status consistency
print("=== Payment Status ===")
print(df["payment_status"].value_counts())

print("\n=== Customer Segment ===")
print(df["customer_segment"].value_counts())

=== Payment Status ===
payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1
Name: count, dtype: int64

=== Customer Segment ===
customer_segment
Student         4
Professional    4
Fresher         3
student         1
Name: count, dtype: int64


## 6. Freshness Check

This check identifies how recent the latest valid order data is compared with today's date.

In [19]:
date_check = pd.to_datetime(df["order_date"], errors="coerce")

latest_date = date_check.max()
today = pd.Timestamp.today().normalize()

days_old = (today - latest_date).days

print("Latest valid order date:", latest_date.date())
print("Today:", today.date())
print("Days since latest order:", days_old)

Latest valid order date: 2026-01-18
Today: 2026-09-11
Days since latest order: 236


## 7. Overall Data Quality Summary

This section summarizes the key data quality issues identified during profiling.

In [20]:
print("===== DATA QUALITY SUMMARY =====")

print("\nCompleteness")
print("Total missing values:", df.isnull().sum().sum())

print("\nUniqueness")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

print("\nValidity")
print("Invalid quantity records:", len(invalid_quantity))
print("Invalid discount records:", len(invalid_discount))
print("Invalid or missing date records:", len(invalid_dates))

print("\nConsistency")
print("Payment status values:")
print(df["payment_status"].unique())

print("Customer segment values:")
print(df["customer_segment"].unique())

print("\nFreshness")
print("Latest valid order date:", latest_date.date())
print("Days since latest order:", days_old)

===== DATA QUALITY SUMMARY =====

Completeness
Total missing values: 3

Uniqueness
Duplicate rows: 1
Duplicate order IDs: 1

Validity
Invalid quantity records: 2
Invalid discount records: 1
Invalid or missing date records: 3

Consistency
Payment status values:
['Paid' 'paid' 'Pending' 'Failed' 'Refunded']
Customer segment values:
['Student' 'Fresher' 'student' 'Professional']

Freshness
Latest valid order date: 2026-01-18
Days since latest order: 236


## 8. Findings and Conclusion

The data profiling identified several data quality issues in the retail orders dataset.

### Key Findings

- Missing values were identified in order_date, city, and discount_pct.
- Duplicate order records were identified.
- Invalid quantity values were identified, including a negative value and a non-numeric value.
- An invalid discount percentage above 100 was identified.
- Invalid or missing order dates were identified.
- Some categorical values require standardization, such as payment status and customer segment capitalization.
- The latest valid order date was checked to assess data freshness.

### Conclusion

The profiling results show that the dataset requires data cleaning and validation before it is used for reporting or KPI calculation. The identified quality issues can be controlled using the rules defined in the Data Quality Contract.